<a href="https://colab.research.google.com/github/SaymaSJ/3-DOF-Cobot_-for_supermarket--Dissertation/blob/main/EasyOCR.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# STEP 1: Install packages
!pip install ultralytics easyocr opencv-python-headless

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 46.0/46.0 kB 2.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.4/1.4 MB 34.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.9/2.9 MB 95.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 66.2/66.2 kB 7.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 183.4/183.4 kB 19.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 972.1/972.1 kB 56.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 295.7/295.7 kB 29.1 MB/s eta 0:00:00


In [ ]:
import torch

# Define the path to your .pt file
model_path = 'best.pt'

# To load a PyTorch model, you typically need to define the model architecture first.
# For demonstration, let's assume it's a simple model, or you have the architecture defined elsewhere.
# If you don't have the model class defined, loading might require more context.

try:
    # Load the model state dictionary
    # If the file contains the entire model (not just state_dict), use torch.load(model_path)
    loaded_model_state_dict = torch.load(model_path)
    print(f"Model state dictionary loaded from '{model_path}' successfully.")
    # You would then typically load this state_dict into an instantiated model:
    # model = YourModelClass() # Replace YourModelClass with your actual model's class
    # model.load_state_dict(loaded_model_state_dict)

    # Print some keys to inspect the loaded data
    print("Keys in the loaded state dictionary:", loaded_model_state_dict.keys())

except FileNotFoundError:
    print(f"Error: The file '{model_path}' was not found. Please ensure it's in the correct directory.")
except Exception as e:
    print(f"An error occurred while loading the model: {e}")

Error: The file 'best.pt' was not found. Please ensure it's in the correct directory.


In [ ]:
# STEP 2: Load YOLOv10n model
from ultralytics import YOLO

model = YOLO('/content/best.pt')

print("Model loaded successfully")

Creating new Ultralytics Settings v0.0.8 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/quickstart#ultralytics-settings.
Model loaded successfully


In [ ]:
# STEP 3: Detect and crop expiry-date regions

import os
import cv2

source = '/content/test_images'
output = '/content/yolo_crops'

os.makedirs(output, exist_ok=True)

image_files = []

for root, dirs, files in os.walk(source):
    for f in files:
        if f.lower().endswith(('.jpg', '.jpeg', '.png', '.jfif')):
            image_files.append(os.path.join(root, f))

print("Total images:", len(image_files))

detected = 0

for path in image_files:

    results = model.predict(
        path,
        conf=0.15,
        verbose=False
    )

    r = results[0]

    if len(r.boxes) == 0:
        continue

    # highest-confidence box
    best_index = int(r.boxes.conf.argmax())

    x1, y1, x2, y2 = map(
        int,
        r.boxes.xyxy[best_index].cpu().numpy()
    )

    img = cv2.imread(path)

    h, w = img.shape[:2]

    # 12% padding
    bw = x2 - x1
    bh = y2 - y1

    px = int(bw * 0.12)
    py = int(bh * 0.12)

    x1 = max(0, x1 - px)
    y1 = max(0, y1 - py)
    x2 = min(w, x2 + px)
    y2 = min(h, y2 + py)

    crop = img[y1:y2, x1:x2]

    filename = os.path.basename(path)

    cv2.imwrite(
        os.path.join(output, filename),
        crop
    )

    detected += 1

print("Detected:", detected)
print("Crops saved in:", output)

Total images: 0
Detected: 0
Crops saved in: /content/yolo_crops


In [ ]:
# STEP 4: EasyOCR with preprocessing

import easyocr
import numpy as np

reader = easyocr.Reader(['en'], gpu=True)

def preprocess_crop(crop):

    # white border
    crop = cv2.copyMakeBorder(
        crop,
        10, 10, 10, 10,
        cv2.BORDER_CONSTANT,
        value=[255,255,255]
    )

    # enlarge 3x
    h, w = crop.shape[:2]

    crop = cv2.resize(
        crop,
        (w*3, h*3),
        interpolation=cv2.INTER_CUBIC
    )

    # grayscale
    gray = cv2.cvtColor(
        crop,
        cv2.COLOR_BGR2GRAY
    )

    # CLAHE
    clahe = cv2.createCLAHE(
        clipLimit=2.0,
        tileGridSize=(8,8)
    )

    enhanced = clahe.apply(gray)

    return enhanced

Progress: |██████████████████████████████████████████████████| 100.0% Complete

Progress: |██████████████████████████████████████████████████| 100.0% Complete

In [ ]:
# STEP 5: Run EasyOCR

import pandas as pd

results_list = []

crop_files = [
    f for f in os.listdir(output)
    if f.lower().endswith(('.jpg','.jpeg','.png','.jfif'))
]

for filename in crop_files:

    path = os.path.join(output, filename)

    crop = cv2.imread(path)

    processed = preprocess_crop(crop)

    text_results = reader.readtext(
        processed,
        detail=0,
        paragraph=False,
        decoder='beamsearch',
        allowlist='0123456789ABCDEFGHIJKLMNOPQRSTUVWXYZabcdefghijklmnopqrstuvwxyz/.-: ',
        contrast_ths=0.1,
        adjust_contrast=0.7,
        text_threshold=0.4,
        low_text=0.3
    )

    text = " ".join(text_results)

    results_list.append({
        'image': filename,
        'ocr_text': text
    })

ocr_df = pd.DataFrame(results_list)

ocr_df.to_csv(
    '/content/ocr_results_final.csv',
    index=False
)

display(ocr_df.head(20))

""


In [ ]:
# STEP 5: Run EasyOCR

import pandas as pd

results_list = []

crop_files = [
    f for f in os.listdir(output)
    if f.lower().endswith(('.jpg','.jpeg','.png','.jfif'))
]

for filename in crop_files:

    path = os.path.join(output, filename)

    crop = cv2.imread(path)

    processed = preprocess_crop(crop)

    text_results = reader.readtext(
        processed,
        detail=0,
        paragraph=False,
        decoder='beamsearch',
        allowlist='0123456789ABCDEFGHIJKLMNOPQRSTUVWXYZabcdefghijklmnopqrstuvwxyz/.-: ',
        contrast_ths=0.1,
        adjust_contrast=0.7,
        text_threshold=0.4,
        low_text=0.3
    )

    text = " ".join(text_results)

    results_list.append({
        'image': filename,
        'ocr_text': text
    })

ocr_df = pd.DataFrame(results_list)

ocr_df.to_csv(
    '/content/ocr_results_final.csv',
    index=False
)

display(ocr_df.head(20))

""


In [ ]:
from ultralytics import YOLO

model = YOLO("/content/best.pt")

print("YOLO loaded")

YOLO loaded


In [ ]:
import os

for root, dirs, files in os.walk("/content/test_images"):
    print(root, "->", len(files), "files")
    if files:
        print(files[:5])

/content/test_images -> 0 files
/content/test_images/content -> 0 files
/content/test_images/content/Expiry-Date-Detection--1 -> 0 files
/content/test_images/content/Expiry-Date-Detection--1/test -> 0 files
/content/test_images/content/Expiry-Date-Detection--1/test/images -> 152 files
['20220526_121551_jpg.rf.35aa8a0c23b1232d5ce5c135555ee313.jpg', '20230920_221009_jpg.rf.35921ef2257ccd08f0b1e93bae928017.jpg', '2621_Flipkart_Input_hres_8903105030045_2ca4f2c2_Ecom_3_jpg.rf.04594c749a209d5e003820b989c360e0.jpg', '20220526_121715_jpg.rf.d80f0b42b4aed3a049be9fe96239326d.jpg', '619_Flipkart_Input_8906035990519_fbb64520_Ecom_2_jpg.rf.170462313219286183b1ef24fdabcd9d.jpg']


In [ ]:
!unzip -q "/content/test_images.zip" -d "/content/test_images"

In [ ]:
!unzip -q "/food_images.zip" -d "/content/expdate"

In [ ]:
import os

print(os.listdir("/content/expdate/Date-Real"))
print(len(os.listdir("/content/expdate/Date-Real/images")))

['annotations.json', 'images']
510


In [ ]:
import json

with open("/content/expdate/Date-Real/annotations.json") as f:
    data = json.load(f)

print(type(data))

if isinstance(data, dict):
    print(data.keys())
else:
    print(data[:2])

<class 'dict'>
dict_keys(['img_00001.jpg', 'img_00002.jpg', 'img_00003.jpg', 'img_00004.jpg', 'img_00005.jpg', 'img_00006.jpg', 'img_00007.jpg', 'img_00008.jpg', 'img_00009.jpg', 'img_00010.jpg', 'img_00011.jpg', 'img_00012.jpg', 'img_00013.jpg', 'img_00014.jpg', 'img_00015.jpg', 'img_00016.jpg', 'img_00017.jpg', 'img_00018.jpg', 'img_00019.jpg', 'img_00020.jpg', 'img_00021.jpg', 'img_00022.jpg', 'img_00023.jpg', 'img_00024.jpg', 'img_00025.jpg', 'img_00026.jpg', 'img_00027.jpg', 'img_00028.jpg', 'img_00029.jpg', 'img_00030.jpg', 'img_00031.jpg', 'img_00032.jpg', 'img_00033.jpg', 'img_00034.jpg', 'img_00035.jpg', 'img_00036.jpg', 'img_00037.jpg', 'img_00038.jpg', 'img_00039.jpg', 'img_00040.jpg', 'img_00041.jpg', 'img_00042.jpg', 'img_00043.jpg', 'img_00044.jpg', 'img_00045.jpg', 'img_00046.jpg', 'img_00047.jpg', 'img_00048.jpg', 'img_00049.jpg', 'img_00050.jpg', 'img_00051.jpg', 'img_00052.jpg', 'img_00053.jpg', 'img_00054.jpg', 'img_00055.jpg', 'img_00056.jpg', 'img_00057.jpg', 'img_

In [ ]:
import json

ann_path = "/content/expdate/Date-Real/annotations.json"

with open(ann_path, "r") as f:
    data = json.load(f)

print("Type:", type(data))

if isinstance(data, dict):
    print("Top-level keys:", data.keys())

    # show first item
    first_key = next(iter(data))
    print("\nFirst key:", first_key)
    print("First value:")
    print(data[first_key])

elif isinstance(data, list):
    print("Number of annotation entries:", len(data))
    print("\nFirst entry:")
    print(data[0])

Type: <class 'dict'>
Top-level keys: dict_keys(['img_00001.jpg', 'img_00002.jpg', 'img_00003.jpg', 'img_00004.jpg', 'img_00005.jpg', 'img_00006.jpg', 'img_00007.jpg', 'img_00008.jpg', 'img_00009.jpg', 'img_00010.jpg', 'img_00011.jpg', 'img_00012.jpg', 'img_00013.jpg', 'img_00014.jpg', 'img_00015.jpg', 'img_00016.jpg', 'img_00017.jpg', 'img_00018.jpg', 'img_00019.jpg', 'img_00020.jpg', 'img_00021.jpg', 'img_00022.jpg', 'img_00023.jpg', 'img_00024.jpg', 'img_00025.jpg', 'img_00026.jpg', 'img_00027.jpg', 'img_00028.jpg', 'img_00029.jpg', 'img_00030.jpg', 'img_00031.jpg', 'img_00032.jpg', 'img_00033.jpg', 'img_00034.jpg', 'img_00035.jpg', 'img_00036.jpg', 'img_00037.jpg', 'img_00038.jpg', 'img_00039.jpg', 'img_00040.jpg', 'img_00041.jpg', 'img_00042.jpg', 'img_00043.jpg', 'img_00044.jpg', 'img_00045.jpg', 'img_00046.jpg', 'img_00047.jpg', 'img_00048.jpg', 'img_00049.jpg', 'img_00050.jpg', 'img_00051.jpg', 'img_00052.jpg', 'img_00053.jpg', 'img_00054.jpg', 'img_00055.jpg', 'img_00056.jpg', 

In [ ]:
!pip install -q easyocr

import easyocr
import json
import os
import pandas as pd

ANN_PATH = "/content/expdate/Date-Real/annotations.json"
IMG_DIR  = "/content/expdate/Date-Real/images"

with open(ANN_PATH, "r") as f:
    annotations = json.load(f)

reader = easyocr.Reader(['en'], gpu=True)

rows = []

for i, (filename, info) in enumerate(annotations.items(), start=1):

    # Build GT from annotated components
    components = {
        item["cls"]: item["transcription"]
        for item in info["ann"]
    }

    year  = components.get("year", "")
    month = components.get("month", "")
    day   = components.get("day", "")

    gt_date = f"{year}-{month}-{day}"

    image_path = os.path.join(IMG_DIR, filename)

    # EasyOCR on complete Date-Real crop
    result = reader.readtext(
        image_path,
        detail=0,
        paragraph=False
    )

    ocr_text = " ".join(result)

    rows.append({
        "image": filename,
        "ground_truth": gt_date,
        "ocr_raw": ocr_text
    })

    if i % 50 == 0:
        print(f"Processed {i}/510")

df = pd.DataFrame(rows)

df.to_csv("/content/expdate_easyocr_raw.csv", index=False)

print("\nFinished.")
print(df.head(10))

Processed 50/510
Processed 100/510
Processed 150/510
Processed 200/510
Processed 250/510
Processed 300/510
Processed 350/510
Processed 400/510
Processed 450/510
Processed 500/510

Finished.
           image ground_truth     ocr_raw
0  img_00001.jpg   2021-05-23  2021.05.23
1  img_00002.jpg   2021-05-16  2021 05 16
2  img_00003.jpg   2022-09-10  2022.09,10
3  img_00004.jpg   2021-05-12  2021.05.12
4  img_00005.jpg   2023-06-01  2023/06/01
5  img_00006.jpg   2021-12-01  01-12-2021
6  img_00007.jpg   2021-03-02            
7  img_00008.jpg     21-11-06    06-11-21
8  img_00009.jpg   2021-03-16  16 03 2021
9  img_00010.jpg   2021-12-01            


In [ ]:
import re
import pandas as pd
from datetime import datetime

df = pd.read_csv("/content/expdate_easyocr_raw.csv")


def normalize_gt(gt):
    """
    Convert ground truth into a comparable canonical representation.
    Keeps 2-digit years as 2 digits when the dataset provides them.
    """
    gt = str(gt).strip()

    parts = re.findall(r'\d+', gt)

    if len(parts) != 3:
        return None

    a, b, c = parts

    # YYYY-MM-DD
    if len(a) == 4:
        return f"{a}-{int(b):02d}-{int(c):02d}"

    # YY-MM-DD
    if len(a) == 2:
        return f"{a}-{int(b):02d}-{int(c):02d}"

    return None


def normalize_ocr(text):
    """
    Parse common expiration-date formats using generic rules.
    Returns canonical date form.
    """
    if pd.isna(text):
        return None

    text = str(text).strip()

    if not text:
        return None

    # Replace common punctuation with spaces
    text = re.sub(r'[.,/\\:_]', ' ', text)

    # Keep numeric groups only
    nums = re.findall(r'\d+', text)

    if len(nums) < 3:
        return None

    # Only use first 3 numeric components
    nums = nums[:3]

    a, b, c = nums

    try:

        # -----------------------
        # YYYY MM DD
        # -----------------------
        if len(a) == 4:
            year = a
            month = int(b)
            day = int(c)

            if 1 <= month <= 12 and 1 <= day <= 31:
                return f"{year}-{month:02d}-{day:02d}"

        # -----------------------
        # DD MM YYYY
        # -----------------------
        if len(c) == 4:
            day = int(a)
            month = int(b)
            year = c

            if 1 <= month <= 12 and 1 <= day <= 31:
                return f"{year}-{month:02d}-{day:02d}"

        # -----------------------
        # 2-digit year formats
        # Examples:
        # GT: 21-11-06
        # OCR: 06-11-21
        #
        # Assume DD-MM-YY when
        # last component is year-like.
        # -----------------------
        if len(a) == 2 and len(b) == 2 and len(c) == 2:

            x = int(a)
            y = int(b)
            z = int(c)

            # DD-MM-YY interpretation
            if 1 <= x <= 31 and 1 <= y <= 12:
                return f"{z:02d}-{y:02d}-{x:02d}"

    except:
        return None

    return None


df["gt_normalized"] = df["ground_truth"].apply(normalize_gt)
df["ocr_normalized"] = df["ocr_raw"].apply(normalize_ocr)

df["correct"] = (
    df["gt_normalized"] == df["ocr_normalized"]
)


total = len(df)
correct = int(df["correct"].sum())
accuracy = correct / total * 100

parsed = df["ocr_normalized"].notna().sum()

print("================================")
print("EasyOCR ExpDate Date-Real Result")
print("================================")
print(f"Total images       : {total}")
print(f"OCR parsed         : {parsed}")
print(f"Exact-date correct : {correct}")
print(f"Exact-date accuracy: {accuracy:.2f}%")
print()

print("First 20 results:")
print(
    df[
        [
            "image",
            "ground_truth",
            "ocr_raw",
            "gt_normalized",
            "ocr_normalized",
            "correct"
        ]
    ].head(20).to_string(index=False)
)

df.to_csv(
    "/content/expdate_easyocr_evaluated.csv",
    index=False
)

EasyOCR ExpDate Date-Real Result
Total images       : 510
OCR parsed         : 344
Exact-date correct : 323
Exact-date accuracy: 63.33%

First 20 results:
        image ground_truth     ocr_raw gt_normalized ocr_normalized  correct
img_00001.jpg   2021-05-23  2021.05.23    2021-05-23     2021-05-23     True
img_00002.jpg   2021-05-16  2021 05 16    2021-05-16     2021-05-16     True
img_00003.jpg   2022-09-10  2022.09,10    2022-09-10     2022-09-10     True
img_00004.jpg   2021-05-12  2021.05.12    2021-05-12     2021-05-12     True
img_00005.jpg   2023-06-01  2023/06/01    2023-06-01     2023-06-01     True
img_00006.jpg   2021-12-01  01-12-2021    2021-12-01     2021-12-01     True
img_00007.jpg   2021-03-02         NaN    2021-03-02           None    False
img_00008.jpg     21-11-06    06-11-21      21-11-06       21-11-06     True
img_00009.jpg   2021-03-16  16 03 2021    2021-03-16     2021-03-16     True
img_00010.jpg   2021-12-01         NaN    2021-12-01           None    Fals

In [40]:
import pandas as pd

df = pd.read_csv("/content/expdate_easyocr_evaluated.csv")

# Show only parsed but incorrect cases
wrong = df[
    (df["ocr_normalized"].notna()) &
    (df["correct"] == False)
]

print("Parsed but incorrect:", len(wrong))

display(
    wrong[
        [
            "image",
            "ground_truth",
            "ocr_raw",
            "gt_normalized",
            "ocr_normalized"
        ]
    ]
)

Parsed but incorrect: 21


,image,ground_truth,ocr_raw,gt_normalized,ocr_normalized
12,img_00013.jpg,2021-10-11,2021.10. !4,2021-10-11,2021-10-04
109,img_00110.jpg,2021-05-19,2021 05.1,2021-05-19,2021-05-01
155,img_00156.jpg,2022-07-27,"2022,07.21",2022-07-27,2022-07-21
157,img_00158.jpg,2021-11-21,2021.11.22,2021-11-21,2021-11-22
162,img_00163.jpg,2022-09-03,2022.03.03,2022-09-03,2022-03-03
167,img_00168.jpg,2022-03-20,2022 03 2 0,2022-03-20,2022-03-02
169,img_00170.jpg,2020-05-12,2020.05.1.2,2020-05-12,2020-05-01
201,img_00202.jpg,2021-06-25,26/06/2021,2021-06-25,2021-06-26
216,img_00217.jpg,2021-09-28,2021/09/23,2021-09-28,2021-09-23
223,img_00224.jpg,2022-08-28,28 03 2022,2022-08-28,2022-03-28


In [41]:
import re
import pandas as pd

df = pd.read_csv("/content/expdate_easyocr_raw.csv")

def normalize_gt(gt):
    if pd.isna(gt):
        return None

    parts = re.findall(r'\d+', str(gt))

    if len(parts) != 3:
        return None

    a, b, c = parts

    if len(a) == 4:
        return f"{a}-{int(b):02d}-{int(c):02d}"

    if len(a) == 2:
        return f"{a}-{int(b):02d}-{int(c):02d}"

    return None


def normalize_ocr_v2(text):
    if pd.isna(text):
        return None

    text = str(text).strip()

    if not text:
        return None

    nums = re.findall(r'\d+', text)

    if not nums:
        return None

    # --------------------------------
    # YYYY MM D D
    # Example: 2022 03 2 0
    #          2020 05 1 2
    # --------------------------------
    if (
        len(nums) >= 4
        and len(nums[0]) == 4
        and len(nums[1]) <= 2
        and len(nums[2]) == 1
        and len(nums[3]) == 1
    ):
        nums = [
            nums[0],
            nums[1],
            nums[2] + nums[3]
        ] + nums[4:]

    if len(nums) < 3:
        return None

    a, b, c = nums[:3]

    try:

        # YYYY-MM-DD
        if len(a) == 4:
            year = int(a)
            month = int(b)
            day = int(c)

            if (
                1900 <= year <= 2100
                and 1 <= month <= 12
                and 1 <= day <= 31
            ):
                return f"{year:04d}-{month:02d}-{day:02d}"

        # DD-MM-YYYY
        if len(c) == 4:
            day = int(a)
            month = int(b)
            year = int(c)

            if (
                1900 <= year <= 2100
                and 1 <= month <= 12
                and 1 <= day <= 31
            ):
                return f"{year:04d}-{month:02d}-{day:02d}"

        # DD-MM-YY -> YY-MM-DD canonical form
        if len(a) <= 2 and len(b) <= 2 and len(c) == 2:
            day = int(a)
            month = int(b)
            year = int(c)

            if (
                1 <= day <= 31
                and 1 <= month <= 12
            ):
                return f"{year:02d}-{month:02d}-{day:02d}"

    except ValueError:
        pass

    return None


df["gt_normalized"] = df["ground_truth"].apply(normalize_gt)
df["ocr_normalized_v2"] = df["ocr_raw"].apply(normalize_ocr_v2)

df["correct_v2"] = (
    df["gt_normalized"] == df["ocr_normalized_v2"]
)

total = len(df)
parsed = df["ocr_normalized_v2"].notna().sum()
correct = df["correct_v2"].sum()

print("Total images       :", total)
print("OCR parsed         :", parsed)
print("Exact-date correct :", correct)
print(f"Exact-date accuracy: {100*correct/total:.2f}%")

print("\nRecovered by parser V2:")

display(
    df[
        (df["correct_v2"] == True) &
        (
            df["gt_normalized"] !=
            df["ocr_raw"].astype(str)
        )
    ][
        [
            "image",
            "ground_truth",
            "ocr_raw",
            "gt_normalized",
            "ocr_normalized_v2"
        ]
    ].tail(30)
)

Total images       : 510
OCR parsed         : 346
Exact-date correct : 328
Exact-date accuracy: 64.31%

Recovered by parser V2:


,image,ground_truth,ocr_raw,gt_normalized,ocr_normalized_v2
430,img_00431.jpg,2021-06-13,2021.06.13,2021-06-13,2021-06-13
431,img_00432.jpg,2022-11-04,2022.11.04,2022-11-04,2022-11-04
432,img_00433.jpg,2021-01-16,2021.01.16,2021-01-16,2021-01-16
433,img_00434.jpg,2020-08-11,2020.08.11,2020-08-11,2020-08-11
434,img_00435.jpg,2022-09-22,"2022.09,22",2022-09-22,2022-09-22
435,img_00436.jpg,2021-04-24,2021. 04.24,2021-04-24,2021-04-24
436,img_00437.jpg,2020-12-01,2020.12.01,2020-12-01,2020-12-01
437,img_00438.jpg,2022-02-04,2022 . 02. 04,2022-02-04,2022-02-04
438,img_00439.jpg,2021-08-04,"2021,08.04",2021-08-04,2021-08-04
440,img_00441.jpg,2021-08-15,2021.08.15,2021-08-15,2021-08-15


In [ ]:
import os

files = os.listdir("/content/test_images")
print("Number of files:", len(files))
print(files[:10])

Number of files: 1
['content']


In [ ]:
!ls -lah /content

total 13M
drwxr-xr-x 1 root root 4.0K Sep  3 04:25 .
drwxr-xr-x 1 root root 4.0K Sep  3 04:21 ..
-rw-r--r-- 1 root root 5.5M Sep  3 04:05 best.pt
drwxr-xr-x 4 root root 4.0K Aug 24 13:27 .config
-rw-r--r-- 1 root root    1 Sep  3 04:08 ocr_results_final.csv
drwxr-xr-x 1 root root 4.0K Aug 24 13:28 sample_data
drwxr-xr-x 3 root root 4.0K Sep  3 04:25 test_images
-rw-r--r-- 1 root root 7.0M Sep  3 04:25 test_images.zip
drwxr-xr-x 2 root root 4.0K Sep  3 04:06 yolo_crops


In [ ]:
results = model.predict(
    source="/content/test_images/content/Expiry-Date-Detection--1/test/images",
    conf=0.15,
    save_crop=False,
    verbose=False
)

print("Processed images:", len(results))

Processed images: 152


In [ ]:
import os
import cv2

image_dir = "/content/test_images/content/Expiry-Date-Detection--1/test/images"
crop_dir = "/content/yolo_crops_improved"

os.makedirs(crop_dir, exist_ok=True)

saved = 0

for r in results:
    if len(r.boxes) == 0:
        continue

    # highest-confidence detection
    best_idx = int(r.boxes.conf.argmax())

    x1, y1, x2, y2 = map(
        int,
        r.boxes.xyxy[best_idx].cpu().numpy()
    )

    img_path = r.path
    img = cv2.imread(img_path)

    h, w = img.shape[:2]

    # 12% padding
    bw = x2 - x1
    bh = y2 - y1

    px = int(bw * 0.12)
    py = int(bh * 0.12)

    x1 = max(0, x1 - px)
    y1 = max(0, y1 - py)
    x2 = min(w, x2 + px)
    y2 = min(h, y2 + py)

    crop = img[y1:y2, x1:x2]

    filename = os.path.basename(img_path)
    save_path = os.path.join(crop_dir, filename)

    cv2.imwrite(save_path, crop)
    saved += 1

print("Crops saved:", saved)
print("Folder:", crop_dir)

Crops saved: 138
Folder: /content/yolo_crops_improved


In [ ]:
import cv2
import os

input_dir = "/content/yolo_crops_improved"
processed_dir = "/content/ocr_preprocessed"

os.makedirs(processed_dir, exist_ok=True)

count = 0

for filename in os.listdir(input_dir):
    if not filename.lower().endswith((".jpg", ".jpeg", ".png", ".jfif")):
        continue

    path = os.path.join(input_dir, filename)
    img = cv2.imread(path)

    # add white border
    img = cv2.copyMakeBorder(
        img, 10, 10, 10, 10,
        cv2.BORDER_CONSTANT,
        value=[255, 255, 255]
    )

    # enlarge 3x
    h, w = img.shape[:2]
    img = cv2.resize(
        img,
        (w * 3, h * 3),
        interpolation=cv2.INTER_CUBIC
    )

    # grayscale
    gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)

    # CLAHE contrast enhancement
    clahe = cv2.createCLAHE(
        clipLimit=2.0,
        tileGridSize=(8, 8)
    )
    enhanced = clahe.apply(gray)

    cv2.imwrite(
        os.path.join(processed_dir, filename),
        enhanced
    )

    count += 1

print("Processed crops:", count)

Processed crops: 138


In [ ]:
!pip install -q easyocr

In [ ]:
import easyocr
import os
import pandas as pd

reader = easyocr.Reader(['en'], gpu=True)

input_dir = "/content/ocr_preprocessed"

rows = []

for filename in os.listdir(input_dir):
    if not filename.lower().endswith((".jpg", ".jpeg", ".png", ".jfif")):
        continue

    path = os.path.join(input_dir, filename)

    texts = reader.readtext(
        path,
        detail=0,
        paragraph=False,
        decoder="beamsearch",
        allowlist="0123456789ABCDEFGHIJKLMNOPQRSTUVWXYZabcdefghijklmnopqrstuvwxyz/.-: ",
        contrast_ths=0.1,
        adjust_contrast=0.7,
        text_threshold=0.4,
        low_text=0.3
    )

    rows.append({
        "image": filename,
        "ocr_text": " ".join(texts)
    })

df = pd.DataFrame(rows)
df.to_csv("/content/ocr_results_improved.csv", index=False)

print("OCR completed:", len(df))
display(df.head(20))

/usr/local/lib/python3.13/dist-packages/easyocr/utils.py:221: RuntimeWarning: overflow encountered in scalar add
  curr.entries[labeling].prTotal += prBlank + prNonBlank
/usr/local/lib/python3.13/dist-packages/easyocr/utils.py:249: RuntimeWarning: overflow encountered in scalar add
  curr.entries[newLabeling].prTotal += prNonBlank
/usr/local/lib/python3.13/dist-packages/easyocr/utils.py:248: RuntimeWarning: overflow encountered in scalar add
  curr.entries[newLabeling].prNonBlank += prNonBlank
/usr/local/lib/python3.13/dist-packages/easyocr/utils.py:249: RuntimeWarning: overflow encountered in scalar add
  curr.entries[newLabeling].prTotal += prNonBlank
/usr/local/lib/python3.13/dist-packages/easyocr/utils.py:248: RuntimeWarning: overflow encountered in scalar add
  curr.entries[newLabeling].prNonBlank += prNonBlank
/usr/local/lib/python3.13/dist-packages/easyocr/utils.py:221: RuntimeWarning: overflow encountered in scalar add
  curr.entries[labeling].prTotal += prBlank + prNonBlank
/u

OCR completed: 138


,image,ocr_text
0,20220526_121551_jpg.rf.35aa8a0c23b1232d5ce5c13...,06.02 EA 13016
1,20230920_221009_jpg.rf.35921ef2257ccd08f0b1e93...,No 1 461 0F al thesV Rs FOR Rs. 730.9 For 30 0...
2,2621_Flipkart_Input_hres_8903105030045_2ca4f2c...,YARDLEY
3,20220526_121715_jpg.rf.d80f0b42b4aed3a049be9fe...,06.02 00:15 FJ
4,619_Flipkart_Input_8906035990519_fbb64520_Ecom...,Vag AICIl Ach TKDIY oc Niax RKIAIL rMice F7600...
5,2551_Flipkart_Input_hres_8908001105446_5f39c91...,Js 450 Lm 12824988 Vap k 9- Detocerg:
6,20230920_213653_jpg.rf.36e1f0bbc2348bb85f0cdc8...,H WwY W M WAHH M
7,KakaoTalk_20220526_185009068_06_jpg.rf.4e0ccdd...,B 4 04.14 23
8,IMG_20230425_205142_jpg.rf.85a5478400429a60699...,EXP DATE 29JAN2024 RVs09
9,123123_jpg.rf.bec0e1e8d66b6a0727140bcd6cd3efa3...,21 09 DABKb .21 Wed


In [ ]:
import pandas as pd

gt = pd.read_csv("/expiry_ocr_ground_truth (1).csv")

gt.loc[
    gt["image"] == "#NAME?",
    "image"
] = "-_jpg.rf.91b04ce0109b2d83149fb9c744a55e1e.jpg"

gt.to_csv("/content/ground_truth_fixed.csv", index=False)

print("Fixed.")

Fixed.


In [ ]:
import pandas as pd
import re

# Load files
gt = pd.read_csv("/content/ground_truth_fixed.csv")
ocr = pd.read_csv("/content/ocr_results_improved.csv")

# Merge by image filename
df = gt.merge(
    ocr[["image", "ocr_text"]],
    on="image",
    how="inner",
    suffixes=("_old", "_new")
)

print("Matched rows:", len(df))

Matched rows: 138


In [ ]:
def normalize_date_text(x):
    if pd.isna(x):
        return ""

    x = str(x).strip().lower()

    # remove spaces
    x = re.sub(r"\s+", "", x)

    # normalize separators
    x = x.replace("-", "/")
    x = x.replace(".", "/")

    return x


df["gt_norm"] = df["ground_truth_date"].apply(normalize_date_text)
df["ocr_norm"] = df["ocr_text_new"].apply(normalize_date_text)

df["correct"] = df.apply(
    lambda r: r["gt_norm"] in r["ocr_norm"]
    if r["gt_norm"] != ""
    else False,
    axis=1
)

correct = df["correct"].sum()
total = len(df)

accuracy = correct / total * 100

print("Correct:", correct)
print("Total:", total)
print(f"OCR exact-date accuracy: {accuracy:.2f}%")

Correct: 4
Total: 138
OCR exact-date accuracy: 2.90%


In [ ]:
import re
import pandas as pd

def extract_date(text):
    if pd.isna(text):
        return ""

    text = str(text).upper()

    # common OCR cleanup
    text = text.replace("O", "0")
    text = text.replace("I", "1")
    text = text.replace("L", "1")

    # normalize separators/spaces
    text = re.sub(r"\s+", " ", text)

    patterns = [
        # YYYY.MM.DD / YYYY-MM-DD / YYYY/MM/DD
        r"\b(20\d{2})[./\-](0?[1-9]|1[0-2])[./\-]([0-3]?\d)\b",

        # DD.MM.YYYY
        r"\b([0-3]?\d)[./\-](0?[1-9]|1[0-2])[./\-](20\d{2})\b",

        # MM.YYYY
        r"\b(0?[1-9]|1[0-2])[./\-](20\d{2})\b",
    ]

    for i, pattern in enumerate(patterns):
        m = re.search(pattern, text)

        if not m:
            continue

        if i == 0:
            y, mo, d = m.groups()
            return f"{y}/{int(mo):02d}/{int(d):02d}"

        elif i == 1:
            d, mo, y = m.groups()
            return f"{y}/{int(mo):02d}/{int(d):02d}"

        elif i == 2:
            mo, y = m.groups()
            return f"{y}/{int(mo):02d}"

    return ""

In [ ]:
def normalize_gt(text):
    if pd.isna(text):
        return ""

    text = str(text).strip()

    # YYYY.MM.DD
    m = re.fullmatch(r"(20\d{2})[./\-](\d{1,2})[./\-](\d{1,2})", text)
    if m:
        y, mo, d = m.groups()
        return f"{y}/{int(mo):02d}/{int(d):02d}"

    # DD.MM.YYYY
    m = re.fullmatch(r"(\d{1,2})[./\-](\d{1,2})[./\-](20\d{2})", text)
    if m:
        d, mo, y = m.groups()
        return f"{y}/{int(mo):02d}/{int(d):02d}"

    # MM.YYYY
    m = re.fullmatch(r"(\d{1,2})[./\-](20\d{2})", text)
    if m:
        mo, y = m.groups()
        return f"{y}/{int(mo):02d}"

    return text.upper().replace(".", "/").replace("-", "/")

In [ ]:
df["parsed_ocr_date"] = df["ocr_text_new"].apply(extract_date)
df["gt_date_norm"] = df["ground_truth_date"].apply(normalize_gt)

df["correct_parsed"] = (
    df["parsed_ocr_date"] == df["gt_date_norm"]
)

correct = df["correct_parsed"].sum()
total = len(df)

print("Correct:", correct)
print("Total:", total)
print(f"Parsed OCR exact-date accuracy: {correct/total*100:.2f}%")

display(
    df[
        ["image", "ground_truth_date", "ocr_text_new",
         "parsed_ocr_date", "correct_parsed"]
    ].head(30)
)

Correct: 2
Total: 138
Parsed OCR exact-date accuracy: 1.45%


,image,ground_truth_date,ocr_text_new,parsed_ocr_date,correct_parsed
0,-_jpg.rf.91b04ce0109b2d83149fb9c744a55e1e.jpg,2023.05.31,2023.05.317144,,False
1,1011_Flipkart_Input_hres_041588_a81b8822_Ecom_...,5-May-22,Net Weight : J8u Gm:: Month 5 MAY Yu2 MRP Rs: ...,,False
2,1193_Flipkart_Input_hres_8906002082124_53146e5...,22-Jan,2 8 0,,False
3,11_jpg.rf.d909896e670b92a03b1cf5389beff701.jpg,2011.01.28,201101.2884 204006271,,False
4,123123_jpg.rf.bec0e1e8d66b6a0727140bcd6cd3efa3...,21.09.21,21 09 DABKb .21 Wed,,False
5,1241_Flipkart_Input_hres_4906227128533_e816c8d...,20.SEP.2021,8 m 8 3,,False
6,1420_Flipkart_Input_hres_8901207035869_c33a375...,UNREADABLE,NaN,,False
7,1439_Flipkart_Input_hres_8906071700806_a7f1daf...,UNREADABLE,NaN,,False
8,1461_Flipkart_Input_8906027780357_106736e3_Eco...,1/10/2021,Arupiol Net Weicht 00g R Uee Pkd 50 / 4 710 / ...,,False
9,1467_Flipkart_Input_hres_8908003948966_da1f92c...,UNREADABLE,Jo a,,False
